In [2]:
import os
import sys
import json
import random
from pathlib import Path
from pprint import pprint

import pandas as pd
from pydantic import BaseModel, Field
from openai import OpenAI

cwd = Path.cwd()
BACKEND_DIR = next((p for p in [cwd, *cwd.parents] if (p / "manage.py").exists()), None)

if BACKEND_DIR is None:
    BACKEND_DIR = Path(r"C:\project_skn\final\Final_project\backend")

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from common import report as report_service

print("BACKEND_DIR:", BACKEND_DIR)
print("OPENAI_API_KEY:", bool(os.getenv("OPENAI_API_KEY")))

BACKEND_DIR: c:\project_skn\final\Final_project\backend
OPENAI_API_KEY: True


In [ ]:
# 입력 데이터 준비
company_dict = {
    "company_description": "LLM 기반 채용 평가 자동화 서비스를 개발하는 회사입니다.",
    "employee_count": 80,
    "team_composition": ["백엔드팀", "AI팀", "서비스기획팀"],
    "employ_style": ["문서 기반 협업", "빠른 실험", "자율적인 문제 해결"],
}

jd_dict = {
    "job_name": "AI 백엔드 개발자",
    "career_level": "주니어",
    "required_skill": ["Python", "Django", "OpenAI API"],
    "preferred_skill": ["LangGraph", "Celery", "Redis"],
    "main_task": "LLM 기반 분석 리포트 생성 파이프라인 개발",
    "hiring_reason": "지원자 분석 자동화 기능 고도화",
    "work_type": "정규직",
}

checklist = [
    "Python 기반 백엔드 개발 경험이 있는가",
    "Django 또는 유사 웹 프레임워크 사용 경험이 있는가",
    "OpenAI API 또는 LLM API 활용 경험이 있는가",
    "비동기 작업 처리 경험이 있는가",
    "협업 과정에서 API 명세를 문서화한 경험이 있는가",
]

resume_dict = {
    "skill": ["Python", "Django", "OpenAI API", "Celery"],
    "experience": [
        {
            "company": "[COMP_NAME_1]",
            "period": "2024.01 ~ 2025.12",
            "role": "백엔드 개발자",
            "description": "Django 기반 API와 비동기 리포트 생성 작업을 개발했습니다.",
        }
    ],
    "self_intoduction": [
        {
            "question": "본인의 강점을 설명해주세요.",
            "answer": "리포트 생성이 느려졌을 때 로그를 확인해 병목 구간을 찾고, Celery 작업 분리와 캐싱을 적용해 처리 시간을 줄였습니다.",
        },
        {
            "question": "팀 프로젝트에서 어려움을 해결한 경험을 작성해주세요.",
            "answer": "프론트엔드와 백엔드의 API 명세 이해가 달라 일정이 지연된 적이 있습니다. 제가 Swagger 문서와 예시 응답을 정리하고 회의를 주도해 요청/응답 구조를 합의했습니다.",
        },
        {
            "question": "지원 동기를 작성해주세요.",
            "answer": "LLM을 활용해 채용 평가 과정을 더 효율적으로 만드는 서비스에 관심이 있습니다. Django와 OpenAI API 기반 개발 역량으로 분석 리포트 품질을 높이고 싶습니다.",
        },
    ],
}

In [ ]:
# STAR 적용 전/후 생성 함수
def run_report_pipeline_without_star(company_dict, jd_dict, checklist, resume_dict):
    """
    STAR 분석을 거치지 않고 기존 생성 단계만 실행한다.
    현재 report.invoke()는 STAR 노드를 타므로, 비교용 baseline은 하위 함수를 직접 호출한다.
    """
    fit_checks = report_service.check_resume_fit(
        resume_summary=resume_dict,
        checklist=checklist,
    )

    fit_feedback = report_service.evaluate_resume_fit_with_feedback(
        resume_info=resume_dict,
        fit_checks=fit_checks,
    )
    fit_checks = report_service._normalize_fit_checks(
        fit_feedback["outputdata"]["checklist"]
    )

    questions = report_service.make_interview_questions(
        resume_summary=resume_dict,
        company_summary=company_dict,
        jd_summary=jd_dict,
        checklist_checks=fit_checks,
    )

    question_feedback = report_service.evaluate_interview_questions_with_feedback(
        resume_info=resume_dict,
        company_info=company_dict,
        jd_info=jd_dict,
        fit_checks=fit_checks,
        questions=questions,
    )
    questions = question_feedback["outputdata"]["questions"]

    report_data = report_service.make_report(
        resume_summary=resume_dict,
        fit_checks=fit_checks,
    )

    report_feedback = report_service.evaluate_report_with_feedback(
        resume_info=resume_dict,
        company_info=company_dict,
        jd_info=jd_dict,
        fit_checks=fit_checks,
        report_data=report_data,
    )
    report_data = report_feedback["outputdata"]
    report_data["question"] = questions

    return {
        "result": report_data,
        "fit_feedback": fit_feedback,
        "question_feedback": question_feedback,
        "report_feedback": report_feedback,
        "resume_used": resume_dict,
    }


def run_report_pipeline_with_star(company_dict, jd_dict, checklist, resume_dict):
    star_resume = report_service.analyze_resume_self_intro_with_star(resume_dict)
    output = run_report_pipeline_without_star(
        company_dict=company_dict,
        jd_dict=jd_dict,
        checklist=checklist,
        resume_dict=star_resume,
    )
    output["star_resume"] = star_resume
    return output

In [5]:
baseline = run_report_pipeline_without_star(
    company_dict=company_dict,
    jd_dict=jd_dict,
    checklist=checklist,
    resume_dict=resume_dict,
)

star_version = run_report_pipeline_with_star(
    company_dict=company_dict,
    jd_dict=jd_dict,
    checklist=checklist,
    resume_dict=resume_dict,
)

print("baseline grade:", baseline["result"].get("overall_grade"))
print("star grade:", star_version["result"].get("overall_grade"))

print("\n[STAR resume]")
pprint(star_version["star_resume"])

baseline grade: B
star grade: A

[STAR resume]
{'experience': [{'company': '[COMP_NAME_1]',
                 'description': 'Django 기반 API와 비동기 리포트 생성 작업을 개발했습니다.',
                 'period': '2024.01 ~ 2025.12',
                 'role': '백엔드 개발자'}],
 'self_intoduction': [{'answer': '리포트 생성의 병목 구간을 파악하기 위해 로그를 확인하고, Celery 작업 '
                                 '분리와 캐싱을 적용하여 처리 시간을 줄인 경험입니다.',
                       'question': '본인의 강점을 설명해주세요.'},
                      {'answer': '팀 프로젝트에서 프론트엔드와 백엔드의 API 명세 차이로 일정이 지연될 때, '
                                 'Swagger 문서 정리와 회의를 통해 요청/응답 구조를 합의하여 문제를 해결한 '
                                 '경험입니다.',
                       'question': '팀 프로젝트에서 어려움을 해결한 경험을 작성해주세요.'},
                      {'answer': 'LLM을 활용한 채용 평가 효율화 서비스에 관심을 가지고, Django와 '
                                 'OpenAI API 기반 개발 역량으로 분석 리포트 품질 향상을 목표로 하는 '
                                 '지원 동기입니다.',
                       'question': '지원 동기를 작성해주세요.'}],
 'skill': ['Python',

In [6]:
# Judge 스키마 정의
class DimensionScore(BaseModel):
    score: int = Field(ge=1, le=5, description="1~5 점수")
    reason: str = Field(description="점수 근거")


class QuestionsJudgeResult(BaseModel):
    grounding: DimensionScore = Field(description="지원서/JD/체크리스트 근거 반영도")
    relevance: DimensionScore = Field(description="직무 및 회사 맥락 관련성")
    coverage: DimensionScore = Field(description="핵심 역량과 우려사항 커버리지")
    answer_quality: DimensionScore = Field(description="모범 답안의 구체성 및 유용성")
    diversity: DimensionScore = Field(description="질문 간 중복 적음과 평가 목적 다양성")
    hallucination_control: DimensionScore = Field(description="근거 없는 사실 추가 억제")
    overall_score: int = Field(ge=1, le=100, description="종합 점수")
    summary: str = Field(description="전체 평가 요약")
    strengths: list[str] = Field(description="강점")
    weaknesses: list[str] = Field(description="약점")


class ReportJudgeResult(BaseModel):
    grounding: DimensionScore = Field(description="지원서 근거 반영도")
    checklist_consistency: DimensionScore = Field(description="체크리스트 결과와 리포트 일관성")
    grade_consistency: DimensionScore = Field(description="등급 산정 일관성")
    insight_quality: DimensionScore = Field(description="강점/우려/검증 포인트의 품질")
    actionability: DimensionScore = Field(description="면접관이 활용 가능한 정도")
    hallucination_control: DimensionScore = Field(description="근거 없는 사실 추가 억제")
    overall_score: int = Field(ge=1, le=100, description="종합 점수")
    summary: str = Field(description="전체 평가 요약")
    strengths: list[str] = Field(description="강점")
    weaknesses: list[str] = Field(description="약점")


class PairwiseJudgeResult(BaseModel):
    baseline_questions: QuestionsJudgeResult
    star_questions: QuestionsJudgeResult
    questions_winner: str = Field(description="'baseline', 'star', 'tie' 중 하나")
    questions_winner_reason: str

    baseline_report: ReportJudgeResult
    star_report: ReportJudgeResult
    report_winner: str = Field(description="'baseline', 'star', 'tie' 중 하나")
    report_winner_reason: str

    final_winner: str = Field(description="'baseline', 'star', 'tie' 중 하나")
    final_reason: str

In [16]:
# LLM-as-a-judge 평가 함수
JUDGE_MODEL = "gpt-4o-mini"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

JUDGE_SYSTEM_PROMPT = """
너는 채용 평가 시스템의 품질을 평가하는 엄격한 LLM judge다.
두 결과물은 같은 회사/JD/checklist에서 생성되었고, 차이는 지원서 자기소개서 답변을 STAR 분석으로 치환했는지 여부다.

평가 원칙:
- 입력 지원서, 회사 정보, JD, 체크리스트에 근거한 내용인지 본다.
- 근거 없는 회사 사실, 지원자 경험, 수치, 성과를 만들면 감점한다.
- 질문지는 실제 면접에서 쓸 수 있는지, 중복이 적은지, 모범 답안이 근거 기반인지 본다.
- 질문지 평가에서 모범 답안은 '정답'이 아니라 면접관 참고용 기대 답변 방향이다.
- 모범 답안에 원본 지원서에 없는 정량 성과, 수치, 세부 성과를 추가하라고 요구하지 마라.
- 모범 답안은 지원서에 이미 있는 근거 안에서 구체적이면 충분하다.
- 리포트는 체크리스트 결과와 등급/요약/강점/우려가 일관되는지 본다.
- STAR 버전이라고 무조건 유리하게 평가하지 말고, 결과물 품질만 비교한다.
- 점수는 1~5 또는 1~100 범위를 엄격히 지킨다.
"""

def judge_pair(company_dict, jd_dict, checklist, original_resume, star_resume, baseline_result, star_result):
    payload = {
        "company_info": company_dict,
        "jd_info": jd_dict,
        "checklist_input": checklist,
        "original_resume": original_resume,
        "star_analyzed_resume": star_resume,
        "baseline_output_without_star": {
            "questions": baseline_result.get("question", []),
            "report": {k: v for k, v in baseline_result.items() if k != "question"},
        },
        "star_output": {
            "questions": star_result.get("question", []),
            "report": {k: v for k, v in star_result.items() if k != "question"},
        },
    }

    response = client.beta.chat.completions.parse(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    "다음 baseline 결과와 STAR 적용 결과를 질문지와 리포트 각각 평가하고 비교해줘.\n\n"
                    + json.dumps(payload, ensure_ascii=False, indent=2)
                ),
            },
        ],
        response_format=PairwiseJudgeResult,
    )
    return response.choices[0].message.parsed.model_dump()

In [17]:
# Judge 실행
judge_result = judge_pair(
    company_dict=company_dict,
    jd_dict=jd_dict,
    checklist=checklist,
    original_resume=resume_dict,
    star_resume=star_version["star_resume"],
    baseline_result=baseline["result"],
    star_result=star_version["result"],
)

pprint(judge_result)

{'baseline_questions': {'answer_quality': {'reason': '질문에 대한 답변이 명확하고 적절히 경험을 '
                                                     '바탕으로 작성되어 있음.',
                                           'score': 4},
                        'coverage': {'reason': '기본적인 기술 요구사항은 고루 다루어졌으나, LLM 및 '
                                               'AI에 관련된 최신 내용은 언급되지 않음.',
                                     'score': 3},
                        'diversity': {'reason': '다양한 질문으로 구성되어 있지만, 일부 경험이 '
                                                '반복되어 다양성이 부족함.',
                                      'score': 3},
                        'grounding': {'reason': '지원자의 경험이 JD에서 요구하는 기술 스택과 맞춰 '
                                                '잘 설명되었으나, 일부 내용에서 구체적인 수치나 '
                                                '성과가 부족함.',
                                      'score': 4},
                        'hallucination_control': {'reason': '지원자의 답변에서 허위 정보가 '
                                              

In [18]:
# 결과 표로 보기
rows = []

for version_key, label in [
    ("baseline_questions", "질문지 - STAR 전"),
    ("star_questions", "질문지 - STAR 후"),
]:
    item = judge_result[version_key]
    rows.append({
        "target": label,
        "overall_score": item["overall_score"],
        "grounding": item["grounding"]["score"],
        "relevance": item["relevance"]["score"],
        "coverage": item["coverage"]["score"],
        "answer_quality": item["answer_quality"]["score"],
        "diversity": item["diversity"]["score"],
        "hallucination_control": item["hallucination_control"]["score"],
        "summary": item["summary"],
    })

for version_key, label in [
    ("baseline_report", "리포트 - STAR 전"),
    ("star_report", "리포트 - STAR 후"),
]:
    item = judge_result[version_key]
    rows.append({
        "target": label,
        "overall_score": item["overall_score"],
        "grounding": item["grounding"]["score"],
        "checklist_consistency": item["checklist_consistency"]["score"],
        "grade_consistency": item["grade_consistency"]["score"],
        "insight_quality": item["insight_quality"]["score"],
        "actionability": item["actionability"]["score"],
        "hallucination_control": item["hallucination_control"]["score"],
        "summary": item["summary"],
    })

df = pd.DataFrame(rows)
df

,target,overall_score,grounding,relevance,coverage,answer_quality,diversity,hallucination_control,summary,checklist_consistency,grade_consistency,insight_quality,actionability
0,질문지 - STAR 전,4,4,4.0,3.0,4.0,3.0,5,"전반적으로 관련 경험이 잘 나타나 있으나, 중복과 구체성 부족이 아쉬운 부분이 있음.",NaN,NaN,NaN,NaN
1,질문지 - STAR 후,5,5,5.0,5.0,5.0,4.0,5,"STAR 방식이 답변의 구조적 명확성과 구체성을 높였으며, 직무와의 연관성도 강화됨.",NaN,NaN,NaN,NaN
2,리포트 - STAR 전,4,4,NaN,NaN,NaN,NaN,5,"전반적인 성과와 적합도가 정의되어 있으나, 일부 깊은 통찰력 부족이 보인다.",5.0,4.0,4.0,4.0
3,리포트 - STAR 후,5,5,NaN,NaN,NaN,NaN,5,"전반적으로 매우 높은 품질의 리포트이며, 지원자의 경험과 기술적 역량이 잘 어필되고 있다.",5.0,5.0,5.0,5.0


In [19]:
import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)

In [20]:
# 질문지 평가
questions_rows = []

for version_key, label in [
    ("baseline_questions", "질문지 - STAR 전"),
    ("star_questions", "질문지 - STAR 후"),
]:
    item = judge_result[version_key]
    questions_rows.append({
        "target": label,
        "overall_score": item["overall_score"],
        "grounding": item["grounding"]["score"],
        "relevance": item["relevance"]["score"],
        "coverage": item["coverage"]["score"],
        "answer_quality": item["answer_quality"]["score"],
        "diversity": item["diversity"]["score"],
        "hallucination_control": item["hallucination_control"]["score"],
        "summary": item["summary"],
    })

questions_df = pd.DataFrame(questions_rows)
questions_df

,target,overall_score,grounding,relevance,coverage,answer_quality,diversity,hallucination_control,summary
0,질문지 - STAR 전,4,4,4,3,4,3,5,"전반적으로 관련 경험이 잘 나타나 있으나, 중복과 구체성 부족이 아쉬운 부분이 있음."
1,질문지 - STAR 후,5,5,5,5,5,4,5,"STAR 방식이 답변의 구조적 명확성과 구체성을 높였으며, 직무와의 연관성도 강화됨."


In [21]:
# 리포트 평가
report_rows = []

for version_key, label in [
    ("baseline_report", "리포트 - STAR 전"),
    ("star_report", "리포트 - STAR 후"),
]:
    item = judge_result[version_key]
    report_rows.append({
        "target": label,
        "overall_score": item["overall_score"],
        "grounding": item["grounding"]["score"],
        "checklist_consistency": item["checklist_consistency"]["score"],
        "grade_consistency": item["grade_consistency"]["score"],
        "insight_quality": item["insight_quality"]["score"],
        "actionability": item["actionability"]["score"],
        "hallucination_control": item["hallucination_control"]["score"],
        "summary": item["summary"],
    })

report_df = pd.DataFrame(report_rows)
report_df

,target,overall_score,grounding,checklist_consistency,grade_consistency,insight_quality,actionability,hallucination_control,summary
0,리포트 - STAR 전,4,4,5,4,4,4,5,"전반적인 성과와 적합도가 정의되어 있으나, 일부 깊은 통찰력 부족이 보인다."
1,리포트 - STAR 후,5,5,5,5,5,5,5,"전반적으로 매우 높은 품질의 리포트이며, 지원자의 경험과 기술적 역량이 잘 어필되고 있다."


In [10]:
# 승자 요약
print("[질문지 비교]")
print("winner:", judge_result["questions_winner"])
print("reason:", judge_result["questions_winner_reason"])

print("\n[리포트 비교]")
print("winner:", judge_result["report_winner"])
print("reason:", judge_result["report_winner_reason"])

print("\n[최종 비교]")
print("winner:", judge_result["final_winner"])
print("reason:", judge_result["final_reason"])

[질문지 비교]
winner: star
reason: STAR 질문지는 구체적인 경험을 잘 반영하고 평가의 깊이가 더 높습니다.

[리포트 비교]
winner: star
reason: STAR 리포트가 지원자의 경험을 더 효과적으로 반영하고 있습니다.

[최종 비교]
winner: star
reason: STAR 방식의 질문과 리포트가 지원자의 경험과 역량을 보다 명확하고 깊이 있게 전달하고 있습니다.


In [22]:
# 실제 인풋과 생성 결과 내용 확인
def print_self_intro_compare(original_resume, star_resume):
    original_items = original_resume.get("self_intoduction") or original_resume.get("self_introduction") or []
    star_items = star_resume.get("self_intoduction") or star_resume.get("self_introduction") or []

    max_len = max(len(original_items), len(star_items))

    for i in range(max_len):
        original_item = original_items[i] if i < len(original_items) else {}
        star_item = star_items[i] if i < len(star_items) else {}

        original_q = original_item.get("question", "") if isinstance(original_item, dict) else ""
        original_a = original_item.get("answer", original_item) if isinstance(original_item, dict) else original_item

        star_q = star_item.get("question", "") if isinstance(star_item, dict) else ""
        star_a = star_item.get("answer", star_item) if isinstance(star_item, dict) else star_item

        print(f"\n{'=' * 80}")
        print(f"[자소서 {i + 1}]")
        print("문항:", original_q or star_q)

        print("\n--- STAR 전 answer ---")
        print(original_a)

        print("\n--- STAR 후 answer ---")
        print(star_a)

print_self_intro_compare(
    original_resume=resume_dict,
    star_resume=star_version["star_resume"],
)


[자소서 1]
문항: 본인의 강점을 설명해주세요.

--- STAR 전 answer ---
리포트 생성이 느려졌을 때 로그를 확인해 병목 구간을 찾고, Celery 작업 분리와 캐싱을 적용해 처리 시간을 줄였습니다.

--- STAR 후 answer ---
리포트 생성의 병목 구간을 파악하기 위해 로그를 확인하고, Celery 작업 분리와 캐싱을 적용하여 처리 시간을 줄인 경험입니다.

[자소서 2]
문항: 팀 프로젝트에서 어려움을 해결한 경험을 작성해주세요.

--- STAR 전 answer ---
프론트엔드와 백엔드의 API 명세 이해가 달라 일정이 지연된 적이 있습니다. 제가 Swagger 문서와 예시 응답을 정리하고 회의를 주도해 요청/응답 구조를 합의했습니다.

--- STAR 후 answer ---
팀 프로젝트에서 프론트엔드와 백엔드의 API 명세 차이로 일정이 지연될 때, Swagger 문서 정리와 회의를 통해 요청/응답 구조를 합의하여 문제를 해결한 경험입니다.

[자소서 3]
문항: 지원 동기를 작성해주세요.

--- STAR 전 answer ---
LLM을 활용해 채용 평가 과정을 더 효율적으로 만드는 서비스에 관심이 있습니다. Django와 OpenAI API 기반 개발 역량으로 분석 리포트 품질을 높이고 싶습니다.

--- STAR 후 answer ---
LLM을 활용한 채용 평가 효율화 서비스에 관심을 가지고, Django와 OpenAI API 기반 개발 역량으로 분석 리포트 품질 향상을 목표로 하는 지원 동기입니다.


In [23]:
def print_questions_compare(baseline_questions, star_questions):
    max_len = max(len(baseline_questions), len(star_questions))

    for i in range(max_len):
        b = baseline_questions[i] if i < len(baseline_questions) else {}
        s = star_questions[i] if i < len(star_questions) else {}

        print(f"\n{'=' * 100}")
        print(f"[질문 {i + 1}]")

        print("\n--- STAR 전 질문 ---")
        print("Q:", b.get("question", ""))
        print("A:", b.get("answer", ""))
        print("Purpose:", b.get("purpose", ""))

        print("\n--- STAR 후 질문 ---")
        print("Q:", s.get("question", ""))
        print("A:", s.get("answer", ""))
        print("Purpose:", s.get("purpose", ""))

baseline_questions = baseline["result"].get("question", [])
star_questions = star_version["result"].get("question", [])

print_questions_compare(baseline_questions, star_questions)


[질문 1]

--- STAR 전 질문 ---
Q: 백엔드 개발자로서 Python 기반 API 개발 경험에 대해 구체적으로 설명해 주실 수 있습니까? 어떤 프로젝트였고, 어떤 기여를 하셨는지요?
A: 저는 [COMP_NAME_1]에서 Django를 사용해 RESTful API를 개발했습니다. 이 API는 채용 평가 과정을 지원하기 위한 것으로, 여러 이해관계자의 요구사항을 반영해 사용자 인증과 데이터 처리 기능을 구현했습니다. 수정 작업을 통해 성능을 크게 개선하였습니다.
Purpose: 지원자의 Python 기반 API 개발 경험과 기여를 평가하기 위함이다.

--- STAR 후 질문 ---
Q: Django 기반 API와 비동기 리포트 생성 작업을 개발했다 말씀하셨습니다. 구체적으로 어떤 문제를 해결하기 위해 어떤 기능을 개발하셨는지 설명해주시겠어요?
A: 저는 Django를 활용해 비즈니스 요구에 맞는 RESTful API를 개발했습니다. 이 과정에서 비동기 처리가 필요했던 리포트 생성 작업에서는 Celery를 사용하여 작업을 분리하고, 처리를 빠르게 하도록 최적화했습니다. 이를 통해 리포트 생성의 병목을 줄이고, 사용자 피드백에 빠르게 대응할 수 있었습니다.
Purpose: 지원자의 기술적 역량과 문제 해결 능력을 심층적으로 평가하기 위함입니다.

[질문 2]

--- STAR 전 질문 ---
Q: Django를 사용하여 작업한 경험에 대해 이야기해 주실 수 있습니까? 어떤 기능을 개발했으며, 그 과정에서의 어려움은 무엇이었고 어떻게 해결했나요?
A: Django를 사용하여 비동기 리포트 생성 API를 개발했습니다. 리포트 생성 속도가 느려지는 문제를 발견하고, 이 문제를 해결하기 위해 Celery를 도입하여 작업을 비동기로 처리하도록 바로잡았습니다.
Purpose: 지원자의 Django 사용 경험과 문제 해결 능력을 평가하기 위함이다.

--- STAR 후 질문 ---
Q: Celery를 사용한 비동기 작업 처리 경험에 대해 구체적으로 말씀해주실 수

In [24]:
def print_report_compare(baseline_report, star_report):
    fields = [
        "overall_grade",
        "overall_summary",
        "candidate_summary",
        "fit_analysis",
        "motive",
        "collaboration",
        "competency_analysis",
        "strength",
        "concern",
        "check_point",
        "final_comment",
    ]

    for field in fields:
        print(f"\n{'=' * 100}")
        print(f"[{field}]")

        print("\n--- STAR 전 ---")
        pprint(baseline_report.get(field))

        print("\n--- STAR 후 ---")
        pprint(star_report.get(field))

baseline_report_only = {k: v for k, v in baseline["result"].items() if k != "question"}
star_report_only = {k: v for k, v in star_version["result"].items() if k != "question"}

print_report_compare(baseline_report_only, star_report_only)


[overall_grade]

--- STAR 전 ---
'B'

--- STAR 후 ---
'A'

[overall_summary]

--- STAR 전 ---
('지원자는 Python, Django, OpenAI API 등 다양한 기술을 보유하고 있으며, 관련 경험을 통해 실력을 입증하였습니다. '
 '또한 팀 프로젝트에서의 협업 경험이 강조되어 협업 능력 또한 긍정적으로 평가됩니다.')

--- STAR 후 ---
('지원자의 기술 스택과 경험이 직무 요구 사항을 매우 잘 충족하고 있습니다. Python, Django, OpenAI API, Celery '
 '등을 활용한 백엔드 개발 경험이 풍부하며, 협업 및 문제 해결 능력이 돋보입니다.')

[candidate_summary]

--- STAR 전 ---
('지원자는 백엔드 개발자로서 Django 기반의 API 및 비동기 리포트 생성 작업에 참여한 경험이 있습니다. 이를 통해 해당 분야에서의 '
 '경력을 쌓았으며, 관련 기술에 대한 이해도가 높은 것으로 보입니다.')

--- STAR 후 ---
('지원자는 Django를 활용한 백엔드 개발 경험이 있으며, LLM 기반 서비스에 대한 관심과 개발 역량이 뚜렷합니다. 또한, 비동기 작업 '
 '처리 및 API 명세 문서화 등의 경험이 있어 프로젝트에서의 기여가 기대됩니다.')

[fit_analysis]

--- STAR 전 ---
('지원자는 Django 기반의 비동기 리포트 생성 작업을 경험하며 복잡한 시스템에서의 문제 해결 능력을 갖추고 있습니다. 이러한 경험은 '
 '해당 직무에 잘 부합하며, 서비스의 효율성을 높이는 데 기여할 수 있습니다.')

--- STAR 후 ---
('지원자는 Django와 OpenAI API를 활용한 개발 경험이 있으며, 이로 인해 LLM 기반 채용 평가 서비스 개발에 적합한 배경을 '
 '가지고 있습니다. 특히 비동기 작업 처리와 API 명세 문서화 경험은 직무와의 적합성을 더욱 높여줍니다.')

[motive]

In [25]:
def print_checklist_compare(baseline_result, star_result):
    b_items = baseline_result.get("checklist", [])
    s_items = star_result.get("checklist", [])
    max_len = max(len(b_items), len(s_items))

    for i in range(max_len):
        b = b_items[i] if i < len(b_items) else {}
        s = s_items[i] if i < len(s_items) else {}

        print(f"\n{'=' * 100}")
        print(f"[체크리스트 {i + 1}]")

        print("내용:", b.get("content") or s.get("content"))
        print("STAR 전 result:", b.get("result"))
        print("STAR 후 result:", s.get("result"))

print_checklist_compare(
    baseline_result=baseline["result"],
    star_result=star_version["result"],
)


[체크리스트 1]
내용: Python 기반 백엔드 개발 경험이 있는가
STAR 전 result: True
STAR 후 result: True

[체크리스트 2]
내용: Django 또는 유사 웹 프레임워크 사용 경험이 있는가
STAR 전 result: True
STAR 후 result: True

[체크리스트 3]
내용: OpenAI API 또는 LLM API 활용 경험이 있는가
STAR 전 result: True
STAR 후 result: True

[체크리스트 4]
내용: 비동기 작업 처리 경험이 있는가
STAR 전 result: True
STAR 후 result: True

[체크리스트 5]
내용: 협업 과정에서 API 명세를 문서화한 경험이 있는가
STAR 전 result: True
STAR 후 result: True


In [26]:
judge_payload = {
    "company_info": company_dict,
    "jd_info": jd_dict,
    "checklist_input": checklist,
    "original_resume": resume_dict,
    "star_analyzed_resume": star_version["star_resume"],
    "baseline_output_without_star": {
        "questions": baseline["result"].get("question", []),
        "report": {k: v for k, v in baseline["result"].items() if k != "question"},
    },
    "star_output": {
        "questions": star_version["result"].get("question", []),
        "report": {k: v for k, v in star_version["result"].items() if k != "question"},
    },
}

pprint(judge_payload)

{'baseline_output_without_star': {'questions': [{'answer': '저는 [COMP_NAME_1]에서 '
                                                           'Django를 사용해 '
                                                           'RESTful API를 '
                                                           '개발했습니다. 이 API는 채용 '
                                                           '평가 과정을 지원하기 위한 '
                                                           '것으로, 여러 이해관계자의 '
                                                           '요구사항을 반영해 사용자 인증과 '
                                                           '데이터 처리 기능을 구현했습니다. '
                                                           '수정 작업을 통해 성능을 크게 '
                                                           '개선하였습니다.',
                                                 'purpose': '지원자의 Python 기반 '
                                                            'API 개발 경험과 기여를 '
                                                            '평가하기 위함이